In [9]:
import pandas as pd

# 需要剔除的状态
exclude_status = ['0.Incomplete', '1.In Progress']

# 读取主表
full_df = pd.read_csv(
    "main_analysis_model.csv",
    parse_dates=["sample_datetime"]
)

# 日期筛选
full_df = full_df[full_df["sample_datetime"] >= "2025-09-01"]

# 只读取业务变量表里需要的列，节省内存
status_df = pd.read_csv(
    "business_analysis_variable_library.csv",
    usecols=["application_id", "application_status"]
)

# 如果 application_id 有重复，保留一条
status_df = status_df.drop_duplicates(subset=["application_id"])

# 关联 application_status
full_df = full_df.merge(
    status_df,
    on="application_id",
    how="left"
)

# 过滤 application_status
full_df = full_df[~full_df["application_status"].isin(exclude_status)]

# 保存
full_df.to_csv("main_analysis_model_filtered.csv", index=False)

print(full_df.shape)
print(full_df["application_status"].value_counts(dropna=False))

(22621, 25)
4.Funded    22621
Name: application_status, dtype: int64


In [ ]:
帮我写一个代码，以application_id为主键，只保留main_analysis_model.csv中存在的application_id，所有的文件

In [ ]:
import os
import pandas as pd

input_dir = r"C:\Users\zhangyuliang02\Desktop\model_analysis_project\INPUT"
main_file = os.path.join(input_dir, "main_analysis_model.csv")
output_dir = os.path.join(input_dir, "filtered_files_by_application_id")
os.makedirs(output_dir, exist_ok=True)


def normalize_application_id(s):
    """
    统一 application_id 格式：
    1. 转字符串
    2. 去掉前后空格
    3. 去掉 .0，例如 12345.0 -> 12345
    4. 空值统一为 None
    """
    s = s.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace(["", "nan", "NaN", "None", "<NA>"], pd.NA)
    return s


print("读取 main_analysis_model.csv ...")

main_df = pd.read_csv(main_file, usecols=["application_id"])

main_df["application_id_norm"] = normalize_application_id(main_df["application_id"])

valid_app_ids = set(main_df["application_id_norm"].dropna().unique())

print(f"主表原始行数: {len(main_df)}")
print(f"主表 application_id 非空行数: {main_df['application_id_norm'].notna().sum()}")
print(f"主表 application_id 去重数量: {len(valid_app_ids)}")
print(f"主表 application_id 示例: {list(valid_app_ids)[:10]}")


for file_name in os.listdir(input_dir):

    if not file_name.lower().endswith(".csv"):
        continue

    if file_name == "main_analysis_model.csv":
        print(f"\n跳过主文件: {file_name}")
        continue

    file_path = os.path.join(input_dir, file_name)

    try:
        print(f"\n处理中: {file_name}")

        df = pd.read_csv(file_path)

        if "application_id" not in df.columns:
            print(f"跳过：无 application_id 字段")
            continue

        raw_rows = len(df)

        df["application_id_norm"] = normalize_application_id(df["application_id"])

        file_total_id_cnt = df["application_id_norm"].notna().sum()
        file_unique_id_cnt = df["application_id_norm"].dropna().nunique()

        matched_mask = df["application_id_norm"].isin(valid_app_ids)
        filtered_df = df[matched_mask].copy()

        matched_rows = len(filtered_df)
        matched_unique_id_cnt = filtered_df["application_id_norm"].dropna().nunique()

        unmatched_rows = raw_rows - matched_rows
        match_rate = matched_rows / raw_rows if raw_rows > 0 else 0

        missing_id_rows = df["application_id_norm"].isna().sum()

        print(f"原始行数: {raw_rows}")
        print(f"本文件 application_id 非空行数: {file_total_id_cnt}")
        print(f"本文件 application_id 去重数量: {file_unique_id_cnt}")
        print(f"匹配成功行数: {matched_rows}")
        print(f"匹配成功 application_id 去重数量: {matched_unique_id_cnt}")
        print(f"未匹配行数: {unmatched_rows}")
        print(f"application_id 为空行数: {missing_id_rows}")
        print(f"行匹配率: {match_rate:.2%}")

        if matched_rows > 0:
            print(
                "匹配到的 application_id 示例:",
                filtered_df["application_id_norm"].dropna().unique()[:10].tolist()
            )

        unmatched_ids = (
            df.loc[~matched_mask, "application_id_norm"]
            .dropna()
            .unique()[:10]
            .tolist()
        )

        if unmatched_ids:
            print(f"未匹配 application_id 示例: {unmatched_ids}")

        filtered_df.drop(columns=["application_id_norm"], inplace=True)

        output_path = os.path.join(output_dir, file_name)
        filtered_df.to_csv(output_path, index=False, encoding="utf-8-sig")

        print(f"保存完成: {output_path}")

    except Exception as e:
        print(f"处理失败 {file_name}: {e}")


print("\n全部处理完成")

读取 main_analysis_model.csv ...
主表原始行数: 65178
主表 application_id 非空行数: 99
主表 application_id 去重数量: 99
主表 application_id 示例: ['2251122', '1202123', '2353518', '2366066', '1876286', '1402231', '1331073', '1889775', '2185224', '1643898']

处理中: business_analysis_variable_library.csv
